# Company Brochure

### Business Challenge

- Create a product that provides the content for a company's brochure for prospective clients, investor and potential recruits.
- You will be given the company's title and primary website address

In [ ]:
# imports
import requests
import json
from requests.models import Response
from IPython.display import display, Markdown
from typing import Optional
from bs4 import BeautifulSoup, Tag

In [ ]:
#constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {
    "Content-Type": "application/json"
}
MODEL = "llama3.1:8b"

In [ ]:
company_name: str = input("Enter the company's name: ")
website_url: str = input("Enter the company's website URL: ")
with_stream = input("Do you want output with stream: (y/n): ")
with_stream = with_stream.strip().lower() == 'y'

if not company_name or not company_name.strip():
    raise ValueError("Company name cannot be empty. Please enter a valid company name.")

if not website_url or not website_url.startswith("http"):
    raise ValueError("Invalid URL. Please enter a valid website URL starting with 'http'.")

In [ ]:
# web scrapper

class WebScraper:
    """A simple web scraper that fetches the content of a webpage and extracts its title."""
    url: str
    title: Optional[str] = None
    body: bytes = b""
    content: str = ""
    links: list[str] = []

    def __init__(self, url: str):
        self.url = url
        self._fetch_content()

    def _fetch_content(self):
        response = requests.get(self.url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        # Safely assign title as a string or None
        self.title = soup.title.string if soup.title and soup.title.string else "No Title Found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input", "button"]):
                irrelevant.decompose()
            self.content = soup.body.get_text(separator="\n", strip=True)

        links: list[str] = [
            href for link in soup.find_all("a", href=True)
            if isinstance(link, Tag) and isinstance((href := link.get("href")) ,str) and href.startswith("http")
        ]
        self.links = links

    def get_content(self) -> str:
        """Returns the content of the webpage."""
        return f"Website Title: {self.title}\n\n Website Content: \n\n{self.content}\n\n"

## Let llama model figure out the relevant links

In [ ]:
system_prompt: str = "You are helful assistant that helps user to create a company brochure. \
    You will be given the content of the company's website and links found on the company's website. \
    You will use the relevant links to gather the information about the links and use the content of the website to create a company brochure. \
    You can use links like About Page, Contact Page, Services, Careers, etc but you will not use links like Privacy Policy, Terms of Service, etc."

system_prompt += "\n\nYou should response in JSON format as the following structure:\n"
system_prompt += """{
    links: [
        {type: "about page", "full_url": "https://www.example.com/about"},
        {type: "careers page", "full_url": "https://www.example.com/careers"}
    ]
}"""

## User Prompt

In [ ]:
def get_user_prompt(website: WebScraper) -> str:
    """Generates the user prompt for the OpenAI API."""
    user_prompt =  f"""
    Here is the list of links found on the company's website: {website.url} -
    Please decide which links are relevant to create a company brochure and which links are not relevant.
    Response with full URL of the links and do not include links that are not relevant, e.g., Privacy Policy, Terms of Service, etc.
    \nLinks (some might be relevant):
    """

    user_prompt += "\n\nYou should response in JSON format as the following structure:\n"
    user_prompt += """{
        links: [
            {type: "about page", "full_url": "https://www.example.com/about"},
            {type: "careers page", "full_url": "https://www.example.com/careers"}
        ]
    }"""
    user_prompt += '\n'.join(website.links)
    return user_prompt

### Model Payload

In [ ]:
def prepare_payload(messages, with_stream=False):
    return {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

## Step 1: Get Relevant Links Using LLAMA

In [ ]:
def get_relevant_links(url: str):
    website = WebScraper(url)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_user_prompt(website)}
    ]
    payload = prepare_payload(messages, False)
    response: Response = requests.post(OLLAMA_API, headers=HEADERS, json=payload)
    if response.status_code == 200:
        content = response.json()['message']['content']
        print(content)

get_relevant_links(website_url)